# MongoDB Contact Manager — Esercitazione Pratica

Progetto sviluppato nell'ambito del **Master in Data Engineering** (ProfessionAI).

Obiettivo: dimostrare l'uso pratico di MongoDB attraverso operazioni CRUD, query avanzate e aggregation pipeline su una collezione di contatti.

**Stack:** MongoDB 7, pymongo, Docker

---

## Schema del documento

```json
{
  "Nome": "string",
  "Cognome": "string",
  "Numero_di_cellulare": "string | array",
  "Società": "string (opzionale)",
  "Data_di_compleanno": "string (opzionale)",
  "Tag": ["string"] (opzionale),
  "Altri_contatti": {
    "Email": "string | array (opzionale)",
    "Indirizzo": "string (opzionale)",
    "Profilo_social": "string (opzionale)"
  },
  "Chiamate_ultimo_mese": "int",
  "Amici_stretti": "bool"
}
```

Il campo `Numero_di_cellulare` può essere stringa o array — questa eterogeneità è intenzionale e dimostra la flessibilità del modello document-based di MongoDB.

## Setup — Connessione e caricamento dataset

In [ ]:
import json
import os
from pymongo import MongoClient
from dotenv import load_dotenv

load_dotenv()

MONGO_URI = os.getenv("MONGO_URI", "mongodb://localhost:27017")
client = MongoClient(MONGO_URI)
db = client["professionai-test"]
collection = db["utenti"]

print(f"Connesso a: {MONGO_URI}")
print(f"Database: {db.name}")

In [ ]:
# Carico il dataset dal file JSON e lo inserisco nella collection
# drop() evita duplicati se la cella viene rieseguita
collection.drop()

with open("../data/utenti.json", "r", encoding="utf-8") as f:
    utenti = json.load(f)

result = collection.insert_many(utenti)
print(f"Inseriti {len(result.inserted_ids)} documenti nella collection 'utenti'")

---
## Punto 1 — Trovare tutti i contatti associati alla società WebCorp

In [ ]:
# Filtro diretto sul campo Società
risultati = list(collection.find({"Società": "WebCorp"}))

print(f"Contatti WebCorp: {len(risultati)}")
for doc in risultati:
    print(f"  - {doc['Nome']} {doc['Cognome']}")

---
## Punto 2 — Identificare i contatti con più di un numero di telefono

Il campo `Numero_di_cellulare` può essere stringa o array. Prima filtro solo i documenti dove è un array (`$type: "array"`), poi verifico con `$expr` che la dimensione sia maggiore di 1.

In [ ]:
risultati = list(collection.find({
    "Numero_di_cellulare": {"$type": "array"},
    "$expr": {"$gt": [{"$size": "$Numero_di_cellulare"}, 1]}
}))

print(f"Contatti con più di un numero: {len(risultati)}")
for doc in risultati:
    print(f"  - {doc['Nome']} {doc['Cognome']}: {doc['Numero_di_cellulare']}")

---
## Punto 3 — Estrarre solo i numeri di telefono dei contatti con tag "lavoro"

`$in` controlla se il valore è presente nell'array `Tag`. La proiezione ritorna solo `Numero_di_cellulare`.

In [ ]:
risultati = list(collection.find(
    {"Tag": {"$in": ["lavoro"]}},
    {"Numero_di_cellulare": 1}
))

print(f"Contatti con tag 'lavoro': {len(risultati)}")
for doc in risultati:
    print(f"  _id: {doc['_id']} → {doc['Numero_di_cellulare']}")

---
## Punto 4 — Riportare nome e cognome dei contatti senza profilo social

`Altri_contatti.Profilo_social` usa la dot notation per accedere a un campo annidato in un documento embedded. `$exists: False` filtra i documenti dove il campo non è presente.

In [ ]:
# Conteggio di controllo: quanti hanno il profilo social
con_social = collection.count_documents({"Altri_contatti.Profilo_social": {"$exists": True}})
totale = collection.count_documents({})
print(f"Totale contatti: {totale} | Con profilo social: {con_social} | Senza: {totale - con_social}")

# Query diretta con $exists: False
risultati = list(collection.find(
    {"Altri_contatti.Profilo_social": {"$exists": False}},
    {"Nome": 1, "Cognome": 1}
))

print("\nContatti senza profilo social:")
for doc in risultati:
    print(f"  - {doc['Nome']} {doc['Cognome']}")

---
## Punto 5 — Contare quanti contatti sono "amici stretti" e quanti no

Aggregation pipeline con `$group` sul campo booleano `Amici_stretti`.

In [ ]:
pipeline = [
    {
        "$group": {
            "_id": {"amici_stretti": "$Amici_stretti"},
            "totale": {"$sum": 1}
        }
    }
]

risultati = list(collection.aggregate(pipeline))
for doc in risultati:
    label = "Amici stretti" if doc["_id"]["amici_stretti"] else "Non amici stretti"
    print(f"  {label}: {doc['totale']}")

---
## Punto 6 — Calcolare la media delle chiamate degli "amici stretti" nell'ultimo mese

`$match` filtra i documenti, `$group` con `$avg` calcola la media.

In [ ]:
pipeline = [
    {"$match": {"Amici_stretti": True}},
    {
        "$group": {
            "_id": None,
            "media_chiamate": {"$avg": "$Chiamate_ultimo_mese"}
        }
    }
]

risultato = list(collection.aggregate(pipeline))[0]
print(f"Media chiamate degli amici stretti: {risultato['media_chiamate']}")

---
## Punto 7 — Aggiungere il numero 345678902 a Simone Azzurri

Il campo `Numero_di_cellulare` di Simone è una stringa. L'update usa una pipeline con `$cond` e `$isArray` per gestire entrambi i casi (stringa o array già esistente), rendendo l'operazione robusta indipendentemente dal tipo del campo.

In [ ]:
# Stato prima dell'update
simone = collection.find_one({"Nome": "Simone", "Cognome": "Azzurri"})
print(f"Prima: {simone['Numero_di_cellulare']}")

# Update con pipeline: se non è array lo trasforma, altrimenti concatena
collection.update_one(
    {"Nome": "Simone", "Cognome": "Azzurri"},
    [
        {
            "$set": {
                "Numero_di_cellulare": {
                    "$cond": {
                        "if": {"$not": {"$isArray": "$Numero_di_cellulare"}},
                        "then": ["$Numero_di_cellulare", "345678902"],
                        "else": {"$concatArrays": ["$Numero_di_cellulare", ["345678902"]]}
                    }
                }
            }
        }
    ]
)

simone = collection.find_one({"Nome": "Simone", "Cognome": "Azzurri"})
print(f"Dopo:  {simone['Numero_di_cellulare']}")

---
## Punto 8 — Inserire il contatto Mary Salgado

`Numero_di_cellulare` viene inserito come array per essere coerente con lo schema e predisposto ad accogliere più numeri.

In [ ]:
nuovo_contatto = {
    "Nome": "Mary",
    "Cognome": "Salgado",
    "Numero_di_cellulare": ["346679933"],
    "Altri_contatti": {
        "Indirizzo": "Via 25 Aprile 3, Firenze"
    }
}

result = collection.insert_one(nuovo_contatto)
print(f"Documento inserito con _id: {result.inserted_id}")

# Verifica
mary = collection.find_one({"Nome": "Mary", "Cognome": "Salgado"})
print(f"\nVerifica: {mary}")

print(f"\nTotale contatti nella collection: {collection.count_documents({})}")